# - Importación librerias:

In [19]:
import os
import re
import string
import nltk
import pandas as pd


# Ensure NLTK resources are downloaded
nltk.download('punkt', halt_on_error=True)
nltk.download('stopwords', halt_on_error=True)
nltk.download('punkt_tab')  # Optional resource

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import SnowballStemmer, PorterStemmer

# Create 'nltk_data' folder if it doesn't exist
if not os.path.exists('nltk_data'):
    os.mkdir('nltk_data')
    # data_path = os.path.abspath('./nltk_data')  # absolute path for nltk_data
    # nltk.data.path.append(data_path)
    # print("NLTK data search paths:", nltk.data.path)

[nltk_data] Downloading package punkt to C:\Users\cescb/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\cescb/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\cescb/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


# -Verificación de las rutas y recursos disponibles:

In [20]:
# Check availability of the 'punkt' tokenizer
try:
    tokenizer = nltk.data.load('tokenizers/punkt/english.pickle')
    print("Resource 'punkt' is available.")
except LookupError:
    print("Resource 'punkt' is NOT available.")

Resource 'punkt' is available.


# - TEXT CLEANING FUNCTION:

In [21]:
def clean_text(text, language='spanish', stemming=1, tokenize_output=1):
    """
    Cleans review text by removing noise for NLP processing.

    Parameters:
    - text (str): Raw text to clean.
    - language (str): Language used for stopwords and stemming ('spanish' or 'english').
    - stemming (int): 1 to apply stemming, 0 to skip.
    - tokenize_output (int): 1 to return token list, 0 to return cleaned string.

    Returns:
    - Cleaned text (list or str).
    """

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www.\S+', '', text)

    # Remove mentions (@user) and hashtags (#topic)
    text = re.sub(r'@\w+|#\w+', '', text)

    # Remove emojis
    emoji_pattern = re.compile("["
                               u"\U0001F600-\U0001F64F"  # Emoticons
                               u"\U0001F300-\U0001F5FF"  # Symbols & pictographs
                               u"\U0001F680-\U0001F6FF"  # Transport & map symbols
                               u"\U0001F1E0-\U0001F1FF"  # Flags
                               u"\U00002500-\U00002BEF"  # Misc symbols
                               "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r'', text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove line breaks and tabs
    text = text.replace('\n', ' ').replace('\t', ' ')

    # Tokenization
    tokens = word_tokenize(text, language=language if language in ['spanish', 'english'] else 'english')

    # Remove stopwords
    try:
        stop_words = set(stopwords.words(language))
    except:
        stop_words = set(stopwords.words('english'))  # Fallback to English

    tokens = [word for word in tokens if word not in stop_words and word.isalpha()]

    # Stemming
    if stemming:
        try:
            stemmer = SnowballStemmer(language)
        except:
            stemmer = PorterStemmer()  # Fallback
        tokens = [stemmer.stem(word) for word in tokens]

    # Return result
    return tokens if tokenize_output else ' '.join(tokens)

## -EXAMPLES: 

In [22]:
# Example 1: Spanish text, stemming + tokenized output
text1 = "¡Hola! Estoy aprendiendo a limpiar texto con #Python y nltk. 😊 Visita https://www.nltk.org/"
tokens1 = clean_text(text1)
print("Example 1 - Tokens with stemming and tokenization:")
print(tokens1)
print()

# Example 2: English text, no stemming, tokenized output
text2 = "This is a test tweet with @cescblancou, #hashtags and a link https://example.com!"
tokens2 = clean_text(text2, language='english', stemming=0, tokenize_output=1)
print("Example 2 - Tokens without stemming:")
print(tokens2)
print()

# Example 3: Spanish text, with stemming, plain string output
text3 = "Los precios subieron un 20% en el último trimestre, según las estadísticas."
cleaned3 = clean_text(text3, language='spanish', stemming=1, tokenize_output=0)
print("Example 3 - Cleaned text with stemming:")
print(cleaned3)
print()

# Example 4: English text, no stemming, plain string output
text4 = "Data science is amazing! Check out https://datascience.com #AI #MachineLearning"
cleaned4 = clean_text(text4, language='english', stemming=0, tokenize_output=0)
print("Example 4 - Cleaned text without stemming:")
print(cleaned4)
print()

# Example 5: Spanish text with emojis, URLs, mentions, numbers
text5 = "¡Feliz cumpleaños Juan! 🎉🎂 Hoy 25/09/2025 es un día especial. Más info en www.fiestas.com #party"
tokens5 = clean_text(text5, language='spanish')
print("Example 5 - Fully cleaned tokens:")
print(tokens5)
print()

Example 1 - Tokens with stemming and tokenization:
['aprend', 'limpi', 'text', 'nltk', 'visit']

Example 2 - Tokens without stemming:
['test', 'tweet', 'link']

Example 3 - Cleaned text with stemming:
preci sub ultim trimestr segun estadist

Example 4 - Cleaned text without stemming:
data science amazing check

Example 5 - Fully cleaned tokens:
['cumpleañ', 'juan', 'hoy', 'dia', 'especial', 'info']



## - SIMULATED DATASET

In [25]:
# Simulated dataset: 10 users, class attended, score (1–5), and review text
data = {
    "user": ["Ana", "Luis", "María", "Carlos", "Elena", "Jorge", "Lucía", "Sofía", "Pedro", "Marta"],
    "class": ["Yoga", "Spinning", "HIIT", "Zumba", "Pilates", "Crossfit", "Yoga", "HIIT", "Zumba", "Spinning"],
    "punctuation": [5, 4, 3, 2, 5, 1, 4, 3, 2, 5],
    "review": [
        "I loved the yoga class 😍. Very relaxing and professional.",
        "What a rhythm in spinning! Although the music was way too loud. 🙉",
        "Too intense for me, but effective. 💪💥 #fitness",
        "Zumba was fun, but the room was too crowded 😓",
        "Perfect for stretching muscles and disconnecting from stress.",
        "I didn't like crossfit, too demanding and no explanations.",
        "HIIT was brutal. I'm exhausted, but I’d definitely do it again!",
        "HIIT fue brutal. Estoy molido, pero repetiría seguro!",
        "The instructor was great, but the sound system wasn't working properly.",
        "Spinning top! Although the bike seat was uncomfortable 😬"
    ]
}

# Create DataFrame
df = pd.DataFrame(data)

df

,user,class,punctuation,review
0,Ana,Yoga,5,I loved the yoga class 😍. Very relaxing and pr...
1,Luis,Spinning,4,What a rhythm in spinning! Although the music ...
2,María,HIIT,3,"Too intense for me, but effective. 💪💥 #fitness"
3,Carlos,Zumba,2,"Zumba was fun, but the room was too crowded 😓"
4,Elena,Pilates,5,Perfect for stretching muscles and disconnecti...
5,Jorge,Crossfit,1,"I didn't like crossfit, too demanding and no e..."
6,Lucía,Yoga,4,"HIIT was brutal. I'm exhausted, but I’d defini..."
7,Sofía,HIIT,3,"HIIT fue brutal. Estoy molido, pero repetiría ..."
8,Pedro,Zumba,2,"The instructor was great, but the sound system..."
9,Marta,Spinning,5,Spinning top! Although the bike seat was uncom...


In [26]:
# Apply cleaning function to the 'review' column
df["clean_review"] = df["review"].apply(lambda x: clean_text(x, language='english', stemming=1, tokenize_output=1))

# Display results
df[["user", "class", "punctuation", "review", "clean_review"]]

,user,class,punctuation,review,clean_review
0,Ana,Yoga,5,I loved the yoga class 😍. Very relaxing and pr...,"[love, yoga, class, relax, profession]"
1,Luis,Spinning,4,What a rhythm in spinning! Although the music ...,"[rhythm, spin, although, music, way, loud]"
2,María,HIIT,3,"Too intense for me, but effective. 💪💥 #fitness","[intens, effect]"
3,Carlos,Zumba,2,"Zumba was fun, but the room was too crowded 😓","[zumba, fun, room, crowd]"
4,Elena,Pilates,5,Perfect for stretching muscles and disconnecti...,"[perfect, stretch, muscl, disconnect, stress]"
5,Jorge,Crossfit,1,"I didn't like crossfit, too demanding and no e...","[didnt, like, crossfit, demand, explan]"
6,Lucía,Yoga,4,"HIIT was brutal. I'm exhausted, but I’d defini...","[hiit, brutal, im, exhaust, definit]"
7,Sofía,HIIT,3,"HIIT fue brutal. Estoy molido, pero repetiría ...","[hiit, fue, brutal, estoy, molido, pero, repet..."
8,Pedro,Zumba,2,"The instructor was great, but the sound system...","[instructor, great, sound, system, wasnt, work..."
9,Marta,Spinning,5,Spinning top! Although the bike seat was uncom...,"[spin, top, although, bike, seat, uncomfort]"
